In [19]:
# Import the necessary libraries for chess analysis
import chess
import chess.pgn
import chess.engine
from chess.engine import Cp
import chess.polyglot
import math
import json
import csv
import time


# board.turn returns True if it's white's turn, False if it's black's turn.
turns = {
    True: "white",
    False: "black"
}
# Open a downloaded .bin file with all openings in polyglot format
books_url = "../material/Book.bin"
books = chess.polyglot.open_reader(books_url)

# ---- Пороги (в единицах win-rate, 0..100) ----
MISS_LOSS_THRESHOLD    = 12   # насколько ход хуже лучшего, чтобы считаться упущением
WINNING_OPPORTUNITY    = 55   # был доступен реальный перевес
SAVING_OPPORTUNITY     = 40   # был доступен реальный шанс сравнять/спастись
NOT_COLLAPSED_FLOOR    = 45   # после хода не рухнул слишком глубоко в минус

# Open a downloaded .tsv file with the names of all openings
def load_opening_book(path):
    opening_book = {}
    with open(path, newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f, delimiter='\t')
        for row in reader:
            opening_book[row['epd']] = (row['eco'], row['name'])
    return opening_book

openings_url = "../material/all.tsv"
openings = load_opening_book(openings_url)

# Open the PGN file for analysis and run the Stockfish.
pgn = open("../material/konteyni_vs_Snowstormme_2026.06.19.pgn")
engine = chess.engine.SimpleEngine.popen_uci(r"/opt/homebrew/Cellar/stockfish/18/bin/stockfish")

# The analyse_game function takes a PGN file and analyzes the game, returning a report with move evaluations.
def analyse_game(pgn):

    # Report template to store game analysis results
    report = {
        "game_id": 'abc123',
        "opening": '',
        "moves": [],
        "eval_history": [],
    }

    openings_state = {"openings_ended": False}

    opportunity = {
        "available_chance": False,
        "chance_for": None,
    }

    game = chess.pgn.read_game(pgn)
    board = game.board()

    info = engine.analyse(board, chess.engine.Limit(depth=16))
    prev_eval = info["score"].white().score(mate_score=10000)

    for move in game.mainline_moves():
        t_move = time.time()

        mover = turns[board.turn]
        move_number = board.fullmove_number

        best_moves = info["pv"]

        board.push(move)
        to_move = turns[board.turn]

        move_info = {
            "move_number": move_number,
            "move": move.uci(),
            "mover": mover,
            "to_move": to_move,
            "mate_in": None,
            "evaluation": None,
            "category": None,
            "fen": board.fen(),
        }
        
        print(board)
        print("\n")

        info = engine.analyse(board, chess.engine.Limit(depth=16))
        eval = info["score"].white().score(mate_score=10000)
        
        category = categorize_move(eval, prev_eval, mover, move, best_moves, board, openings_state, report, opportunity)
        move_info["category"] = category


        print(f"{mover} made a move: {move.uci()}. Evaluation: {prev_eval} -> {eval}: {category}. {to_move} to move.")
        print(f"move took {time.time() - t_move:.2f}s")
        print("\n")
        print("\n")

        prev_eval = eval

        if info["score"].is_mate():
            move_info["mate_in"] = info['score'].white().mate()

        move_info["evaluation"] = eval

        report["moves"].append(move_info)
        report["eval_history"].append(eval)

    return report

def categorize_move(eval, prev_eval, mover, move, best_moves, board, openings_state, report, opportunity):
    current_winrate = calculate_winrate(eval)
    prev_winrate = calculate_winrate(prev_eval)

    if is_book_move(board, report) and not openings_state['openings_ended']:
        return "Book"
    else:
        openings_state['openings_ended'] = True

    if is_forced_move(report):
        return "Forced"

    mover_curr_wr = current_winrate if mover == "white" else 100 - current_winrate

    board_before = board.copy()
    board_before.pop()

    if best_moves and is_miss(board_before, move, best_moves[0], mover, mover_curr_wr, opportunity):
        opportunity['available_chance'] = False
        opportunity['chance_for'] = None
        return "Miss"

    return classic_moves(move, mover, best_moves, current_winrate, prev_winrate, opportunity)

def is_book_move(board, report):
    last_move = board.pop()

    continuations = []
    for continuation in books.find_all(board):
        continuations.append(continuation.move)
    board.push(last_move)

    opening = openings.get(board.epd())
    if opening != None:
        report['opening'] = opening[1]

    book = last_move in continuations
    return book

def is_forced_move(report):
    if len(report['moves']) == 0:
        return False
    board = chess.Board(report['moves'][-1]["fen"])
    forced = board.legal_moves.count() == 1
    return forced


# def is_miss(opportunity, mover, current_winrate):
#     if mover != opportunity['chance_for'] or not opportunity['available_chance']:
#         return False
    
#     temp_winrate = current_winrate if mover == "white" else 100-current_winrate
#     diff = opportunity["peak_winrate"] - temp_winrate
#     miss = diff >= 10 

#     if diff > 2: 
#         opportunity['available_chance'] = False
#         opportunity['peak_winrate'] = None

#     return miss


def winrate_pov(cp_white, mover):
    wr_white = calculate_winrate(cp_white)
    return wr_white if mover == "white" else 100 - wr_white

def evaluate_after_move(board_before, move, depth=15):
    board_before.push(move)
    score = engine.analyse(board_before, chess.engine.Limit(depth=depth))["score"].white().score(mate_score=10000)
    board_before.pop()
    return score

def is_miss(board_before, played_move, best_move, mover, current_winrate, opportunity):
    if not opportunity['available_chance'] or mover != opportunity['chance_for'] or played_move == best_move:
        return False

    best_cp = evaluate_after_move(board_before, best_move)
    wr_best = winrate_pov(best_cp, mover)
    wr_played = current_winrate
    loss = wr_best - wr_played

    # Условие 1 — была ли реальная возможность: победить ИЛИ спастись
    opportunity_existed = wr_best >= WINNING_OPPORTUNITY or wr_best >= SAVING_OPPORTUNITY
    # Условие 2 — отдал ли значимую долю
    gave_up_enough = loss >= MISS_LOSS_THRESHOLD
    # Условие 3 — не рухнул слишком глубоко после хода
    not_collapsed = wr_played >= NOT_COLLAPSED_FLOOR

    return opportunity_existed and gave_up_enough and not_collapsed

# The classic_moves function categorizes the move based on its quality compared to the best move and the change in winrate. 
# Which does not include special moves such as forced, book, , miss, great, brilliant moves.
def classic_moves(move, mover, best_moves, current_winrate, prev_winrate, opportunity):

    if mover == "white":
        winrate_change = prev_winrate - current_winrate
    else:
        winrate_change = current_winrate - prev_winrate
    
    if move == best_moves[0]:
        return "Best"
    elif winrate_change <= 2:
        return "Excellent"
    elif winrate_change <= 5:   
        return "Good"
    elif winrate_change <= 10:
        return "Inaccuracy"
    elif winrate_change <= 20:
        opportunity["available_chance"] = True
        opportunity["chance_for"] = "black" if mover == "white" else "white"
        return "Mistake"
    else:
        opportunity["available_chance"] = True
        opportunity["chance_for"] = "black" if mover == "white" else "white"
        return "Blunder"


def calculate_winrate(x):
    y = 50 + 50 * ( 2 / (1 + math.e**(-0.004 * x)) -1)
    return y

print(json.dumps(analyse_game(pgn), indent=2))

engine.quit()

r n b q k b n r
p p p p p p p p
. . . . . . . .
. . . . . . . .
. . . . P . . .
. . . . . . . .
P P P P . P P P
R N B Q K B N R


white made a move: e2e4. Evaluation: 47 -> 37: Book. black to move.
move took 0.28s




r n b q k b n r
p p p p . p p p
. . . . p . . .
. . . . . . . .
. . . . P . . .
. . . . . . . .
P P P P . P P P
R N B Q K B N R


black made a move: e7e6. Evaluation: 37 -> 47: Book. white to move.
move took 0.19s




r n b q k b n r
p p p p . p p p
. . . . p . . .
. . . . . . . .
. . . . P . . .
. . . . . N . .
P P P P . P P P
R N B Q K B . R


white made a move: g1f3. Evaluation: 47 -> 35: Book. black to move.
move took 0.20s




r n b q k b n r
p p p . . p p p
. . . . p . . .
. . . p . . . .
. . . . P . . .
. . . . . N . .
P P P P . P P P
R N B Q K B . R


black made a move: d7d5. Evaluation: 35 -> 33: Book. white to move.
move took 0.18s




r n b q k b n r
p p p . . p p p
. . . . p . . .
. . . P . . . .
. . . . . . . .
. . . . . N . .
P P P P . P P P
R N B Q K B . R


In [17]:
fen = "4Q3/K7/8/3B4/5Q1p/7P/5P1k/8 b - - 2 56"
board = chess.Board(fen)
print(board.legal_moves.count())

2


In [3]:
fen = "rnbqkbnr/pp1ppppp/8/2p5/4P3/8/PPPP1PPP/RNBQKBNR w KQkq - 0 2"
board = chess.Board(fen)

openings_url = "baron30.bin"
openings = chess.polyglot.open_reader(openings_url)

# for hehe in openings.find_all(board):
#     print(hehe.move)

print(board.epd())

rnbqkbnr/pp1ppppp/8/2p5/4P3/8/PPPP1PPP/RNBQKBNR w KQkq -


In [2]:
print(len(openings))  # сколько всего записей загрузилось
print(list(openings.items())[:3])  # первые три пары ключ-значение, глазами оценить структуру

3733
[('rnbqkbnr/pppppppp/8/8/8/7N/PPPPPPPP/RNBQKB1R b KQkq -', ('A00', 'Amar Opening')), ('rnbqkbnr/ppp2ppp/8/3pp3/5P2/6PN/PPPPP2P/RNBQKB1R b KQkq -', ('A00', 'Amar Opening: Paris Gambit')), ('rn1qkbnr/ppp2ppp/8/3p4/8/6PB/PPPPP3/RNBQ1RK1 b kq -', ('A00', 'Amar Opening: Paris Gambit, Gent Gambit'))]


In [5]:
calculate_winrate(70) - calculate_winrate(-40)

10.083641310897491